# Business Entity Resolution — Amazon ML Challenge

End-to-end pipeline: **exploration → leakage-safe validation → high-recall blocking → pair features → LightGBM matcher → macro-F0.5 decision policy → test predictions → validated submission files.**

The reusable logic lives in the `ber` package next to this notebook (`src/ber/`). The notebook runs every stage step by step with diagnostics; the command line (`python src/run_pipeline.py run --data-dir <student_resource>`) runs the identical code for the reproducible submission package.

| Stage | Module | What it does |
|---|---|---|
| Load + checks | `ber.data` | explicit-tab TSV reader (no NA coercion, no quote parsing), ground-truth parsing and contract checks |
| Normalisation | `ber.normalize` | accent folding, legal-form and abbreviation canonicalisation, DBA / trade-name splitting, phonetic skeletons, postal code / house number / locality extraction — one code path for every country |
| Blocking | `ber.blocking` | union of 7 blockers (char TF-IDF kNN on name, address and name+address; rare shared token; phonetic and no-space keys; postal code + name; acronyms), run per target source, then pruned per S1 entity |
| Features | `ber.features` | ~85 features: string similarities, token/IDF overlaps, numeric and postal agreement, missingness, blocker provenance, and *competition* features (rank and margin of a pair among an entity's candidates and among a target's claimants) |
| Matcher | `ber.model` | LightGBM (MIT licence), stratified group K-fold over S1 entities, isotonic calibration |
| Decision | `ber.decision` | global threshold, per-source thresholds, or expected-F0.5 top-k per entity, each with or without "one S1 per target"; chosen by cross-fitting on out-of-fold predictions |
| Output | `ber.submission` | writes both TSVs exactly as specified and re-checks every submission rule |

**Where this departs from the original plan, and why**

1. **The metric is macro F0.5 per Source-1 entity, singletons included.** Every choice (thresholds, policy family, exclusivity) is tuned on that number, never on pairwise accuracy. An empty prediction is a first-class answer: a correct one is worth a full 1.0.
2. **The matcher trains on the exact candidate distribution it scores at test time** (the full pruned candidate set) instead of a re-sampled 1:3–1:10 positive:negative mix. Re-sampling distorts probabilities, which would break the thresholds and the expected-F0.5 policy.
3. **Thresholds are validated honestly.** Policies are picked by cross-fitting: tuned on four folds' out-of-fold predictions, scored on the fifth.
4. **France / open-set countries.** Accents are folded rather than deleted (a `[^a-z0-9]` filter turns "Société" into "soci t"), the legal-form lexicon covers SARL/SAS/EURL/GmbH/… alongside Inc/LLC/Pvt Ltd, nothing branches on the country value, and country enters the model only as "the two labels agree". A leave-one-country-out test estimates performance on a country the model never saw.
5. **Competition features.** Whether a pair is the best candidate for its entity *and* the best claimant for its target is one of the strongest precision signals, so it is computed explicitly instead of hoping a pairwise model infers it.
6. **Fair play.** Only the provided files are used. The lexicons are hand-written generic language rules; there is no geocoding, API, external database or internet lookup. The final model is LightGBM (MIT licence, a few MB).

## 0. Setup
Set `STUDENT_RESOURCE_DIR` if auto-detection fails (it searches the working directory and its parents for `dataset/train/train_source1.tsv`; the `BER_DATA_DIR` environment variable also works). Outputs go to `student_resource/output/`, the model and validation reports to `student_resource/artifacts/`, exploration tables to `student_resource/analysis/`.

In [ ]:
import os, sys, time, json, platform, zipfile
from pathlib import Path

# ------------------------------------------------------------------ configuration
STUDENT_RESOURCE_DIR = None      # folder that contains dataset/ and utils/; None = auto-detect
RUN_BLOCKING_SWEEP = False       # optional: recall vs candidate count for a few K settings (slower)
RUN_COUNTRY_HOLDOUT = True       # unseen-country stress test (the offline proxy for France)
SEED = 42


def _find_src() -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents][:4]:
        for cand in (base, base / "src", base / "business_entity_resolution" / "src",
                     base / "code" / "business_entity_resolution" / "src"):
            if (cand / "ber" / "__init__.py").exists():
                return cand
    for p in here.rglob("ber/__init__.py"):
        return p.parent.parent
    if Path("/kaggle/working").exists():
        for p in Path("/kaggle/working").rglob("ber/__init__.py"):
            return p.parent.parent
    raise FileNotFoundError("Cannot find src/ber - start Jupyter from the project folder or from src/.")


SRC_DIR = _find_src()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm, rapidfuzz, sklearn

import ber
from ber import explore, plots
from ber.config import Config
from ber.data import check_ground_truth, load_split, resolve_dataset_dir
from ber.normalize import examples_table, normalize_country
from ber.index import build_index
from ber.blocking import generate_candidates
from ber.features import build_features
from ber.model import make_folds, train_matcher
from ber.decision import policy_mask, predictions_from_mask, tune_policy
from ber.metrics import blocking_report, evaluation_report
from ber.pipeline import (PreparedSplit, TrainedPipeline, country_holdout_eval, country_map, explore_split,
                          label_pairs, predict, write_outputs)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
plots.use_style()

DATASET_DIR = resolve_dataset_dir(STUDENT_RESOURCE_DIR)
ROOT = DATASET_DIR.parent
OUTPUT_DIR, ARTIFACT_DIR, ANALYSIS_DIR = ROOT / "output", ROOT / "artifacts", ROOT / "analysis"
for d in (OUTPUT_DIR, ARTIFACT_DIR, ANALYSIS_DIR):
    d.mkdir(parents=True, exist_ok=True)

cfg = Config()
cfg.model.seed = SEED
T_START = time.time()
print(f"ber {ber.__version__} | python {platform.python_version()} | pandas {pd.__version__} | "
      f"lightgbm {lightgbm.__version__} | rapidfuzz {rapidfuzz.__version__} | scikit-learn {sklearn.__version__}")
print("dataset :", DATASET_DIR)
print("outputs :", OUTPUT_DIR)


## 1. Load the data and check the ground-truth contract
Every file is read with an explicit tab separator, as strings, with `keep_default_na=False` (a business called "NA Traders" must not become NaN) and with quote parsing off (a stray `"` in a name cannot swallow the following lines). The row count is cross-checked against the raw line count.

In [ ]:
train = load_split(DATASET_DIR, "train")
test = load_split(DATASET_DIR, "test")
print("train:", train.summary())
print("test :", test.summary())
display(check_ground_truth(train))
display(train.s1.head(3), train.s2.head(3), train.s3.head(3))


## 2. Exploration — quantify the data before modelling
Each table below feeds a concrete design decision (summarised at the end of this section). All tables are also written to `analysis/`.

### 2.1 Countries — an open set

In [ ]:
display(explore.overview(train), explore.overview(test))
train_countries = {normalize_country(c) for c in pd.concat([train.s1, train.s2, train.s3])["country"]} - {""}
test_countries = {normalize_country(c) for c in pd.concat([test.s1, test.s2, test.s3])["country"]} - {""}
unseen_countries = sorted(test_countries - train_countries)
print("country labels only in test:", unseen_countries or "none")
display(explore.country_table(test))


### 2.2 Match cardinality (how many S2/S3 records per S1 entity)

In [ ]:
card = explore.cardinality(train)
display(card["distribution"].round(3), card["per_source"].round(3), card["stats"].round(3).to_frame())
plots.grouped_bars(card["distribution"], "Matches per Source-1 entity (share of entities)")
plt.show()


### 2.3 Duplicate structure and country consistency of true matches
If no target is ever linked to two S1 entities, a "one S1 per target" constraint is plausible; the decision stage still tests it on validation data. Cross-country true pairs would make a country filter unsafe (it is never used as a filter anyway).

In [ ]:
usage = explore.target_usage(train)
display(usage)
multi_linked = int(usage.loc[usage["S1 entities linked"].map(lambda v: isinstance(v, (int, np.integer)) and v > 1), "targets"].sum())
print(f"targets linked to more than one S1 entity: {multi_linked}")
consistency = explore.country_consistency(train)
display(consistency)


### 2.4 Field completeness per source and country
Missing addresses mean name features must carry the decision on their own; landmark-style addresses call for token overlap rather than exact matching; missing postal codes mean postal agreement can only be *evidence*, never a requirement.

In [ ]:
idx_train = build_index(train, cfg.blocking)   # normalised views + TF-IDF matrices, reused by blocking and features
display(explore.completeness(idx_train))


### 2.5 How similar are true matches? (vs random pairs from the same target source)

In [ ]:
sim = explore.similarity_profile(idx_train, train.truth, seed=SEED)
feats = ["name_char_cos", "addr_char_cos", "name_token_set"]
plots.compare_hist(sim, feats, title="True matches vs random same-source pairs")
plt.show()
q = sim.groupby(["label", "target_source"])[feats].quantile([0.05, 0.5]).unstack().round(3)
display(q)
display(sim[sim["label"] == "match"].groupby("country")[feats].median().round(3))


### 2.6 Normalisation — what the matcher actually compares
Random names and addresses from the data (including records from test-only countries, if any) with their normalised views: `core` drops legal forms and stop words, `skeleton` is a phonetic key (Agarwal / Aggarwal / Agrawal collide), `alternates` holds DBA / trade-name parts.

In [ ]:
unseen_rows = pd.concat([test.s2, test.s3])
unseen_rows = unseen_rows[unseen_rows["country"].map(normalize_country).isin(unseen_countries)]
name_examples = (train.s1["business_name"].sample(4, random_state=SEED).tolist()
                 + train.s3["business_name"].sample(3, random_state=SEED).tolist()
                 + unseen_rows["business_name"].head(3).tolist())
addr_examples = (train.s1["business_address"].sample(3, random_state=SEED).tolist()
                 + train.s3["business_address"].sample(3, random_state=SEED).tolist()
                 + unseen_rows["business_address"].head(3).tolist())
display(examples_table(name_examples, "name"))
display(examples_table(addr_examples, "address"))


### 2.7 What the exploration implies

In [ ]:
tables = explore_split(train, cfg, ANALYSIS_DIR, index=idx_train)   # writes every table to analysis/
stats = card["stats"]
singleton_share = card["distribution"].loc["0 (singleton)", "all"]
cross_country = int(consistency["different"].sum()) if "different" in consistency else 0
missing_addr = idx_train.t["addr_missing"].mean()
print(f"* {singleton_share:.1%} of Source-1 entities are singletons: 'no match' must be a normal output, "
      f"and false merges on them cost a full point each.")
print(f"* Up to {int(stats['max matches per S1'])} matches per entity; {stats['S1 with >1 match in the same source']:.1%} "
      f"have several in one source: score pairs independently, never force top-1.")
print(f"* {multi_linked} targets are shared by several entities: 'exclusive targets' is "
      f"{'plausible' if multi_linked == 0 else 'questionable'} (the decision stage tests it on validation data).")
print(f"* {cross_country} true pairs carry different country labels: country is a feature ('labels agree'), never a filter.")
print(f"* {missing_addr:.1%} of S2/S3 addresses are missing: name-only evidence must be enough in those cases.")
print(f"* Countries seen only in test: {unseen_countries or 'none'} - see the unseen-country stress test in section 8.")
print(f"(saved {len(tables)} tables to {ANALYSIS_DIR})")


## 3. Candidate generation (blocking)
Goal: recall first. Seven independent blockers are unioned; the k-nearest-neighbour blockers run **separately for S2 and S3** so one source cannot crowd the other out. The union is then pruned per (S1 entity, target source): a pair survives if it is in the top 30 by a cheap name+address score, or the top 10 by name alone, or the top 5 by address alone, or it came from an exact key. **This pruned set is exactly what the matcher scores and what goes into `candidate_pairs.tsv`.** Country is never used to filter.

In [ ]:
cand_train = generate_candidates(idx_train, cfg.blocking)
countries_train = country_map(train)
brep = blocking_report(cand_train, train.truth, idx_train.n_s1, idx_train.n_t, countries_train, cfg.decision.beta)
display(brep["headline"].round(4).to_frame())
for name, t in brep["tables"].items():
    print(name)
    display(t.round(4))


True pairs that blocking misses can never be recovered by the matcher — this list is where blocking improvements come from.

In [ ]:
missed = explore.pair_view(brep["missed"], train)
print(f"{len(missed)} true pairs are outside the candidate set")
display(missed.head(20))


In [ ]:
if RUN_BLOCKING_SWEEP:
    from dataclasses import replace
    rows = []
    for name_k, comb_k, addr_k, cap in [(10, 10, 5, 15), (25, 25, 10, 30), (50, 50, 20, 60)]:
        bcfg = replace(cfg.blocking, name_k=name_k, combined_k=comb_k, address_k=addr_k, max_candidates_per_source=cap)
        h = blocking_report(generate_candidates(idx_train, bcfg, verbose=False), train.truth,
                            idx_train.n_s1, idx_train.n_t)["headline"]
        rows.append({"name_k": name_k, "combined_k": comb_k, "address_k": addr_k, "cap per source": cap,
                     "pairs": int(h["candidate pairs"]), "per S1": h["candidates per S1 (mean)"],
                     "pair recall": h["pair recall (true pairs retrieved)"],
                     "F0.5 upper bound": h["macro F0.5 upper bound (oracle matcher)"]})
    display(pd.DataFrame(rows).round(4))


## 4. Pair features
One row per candidate pair. Missing information is NaN (LightGBM routes it natively) rather than a fake zero, so "addresses disagree" and "one address is missing" stay different signals.

In [ ]:
X_train = build_features(idx_train, cand_train)
y_train = label_pairs(cand_train, train.truth)
print(f"positives: {int(y_train.sum()):,} of {len(y_train):,} candidate pairs ({y_train.mean():.2%})")
separation = pd.DataFrame({"median | match": X_train[y_train == 1].median(),
                           "median | non-match": X_train[y_train == 0].median(),
                           "missing rate": X_train.isna().mean()})
display(separation.round(3))


## 5. Matcher — LightGBM with grouped, stratified cross-validation
Folds are made of **Source-1 entities** (never of pairs), grouped by connected components of the ground-truth graph and stratified by country × match cardinality. Test-time scores average the five fold models; an isotonic calibrator is fit on the out-of-fold scores (and cross-fitted for the validation numbers).

In [ ]:
country_norm = {k: normalize_country(v) for k, v in countries_train.items()}
s1_fold = make_folds(train.truth, country_norm, cfg.model.n_folds, cfg.model.seed)
pair_fold = cand_train["s1_id"].map(s1_fold).to_numpy()
display(pd.crosstab(pd.Series(s1_fold, name="fold"), pd.Series(country_norm, name="country")))
matcher, oof_raw, oof_cal = train_matcher(X_train, y_train, pair_fold, cfg.model)
display(pd.Series(matcher.cv_info, name="value").round(5).to_frame())
plots.importance_bars(matcher.importance["gain_share"])
plt.show()
plots.reliability(oof_cal, y_train)
plt.show()


## 6. Decision policy — optimise macro F0.5 directly
Candidates are scored independently; the policy then turns probabilities into one list per entity. Families: a global threshold, separate S2 / S3 thresholds, and **expected-F0.5 top-k** (for each entity, pick the top-k set with the highest expected F0.5 under the calibrated probabilities — k = 0 is always allowed, which is what protects singletons). Each family is tried with and without "one S1 per target". The reported "cross-fitted" score tunes on four folds and scores the fifth, so it is an honest estimate.

In [ ]:
oof = cand_train[["s1_id", "t_id", "src"]].copy()
oof["p_raw"], oof["p"], oof["y"], oof["fold"] = oof_raw, oof_cal, y_train, pair_fold
policy, policy_summary, crossfit_f = tune_policy(oof, train.truth, s1_fold, cfg.decision)
display(policy_summary.round(5))
plots.threshold_curve(policy_summary.attrs["full"], policy.threshold if policy.kind == "threshold" else None)
plt.show()
oof_mask = policy_mask(oof, policy, cfg.decision)
report = evaluation_report(predictions_from_mask(oof, oof_mask), train.truth, list(train.truth), countries_train,
                           cfg.decision.beta)
display(report["headline"].round(4).to_frame())
for name, t in report["tables"].items():
    print(name)
    display(t.round(4))


## 7. Error analysis
The most expensive errors are false positives on true singletons (each costs a full 1.0), then high-scoring wrong pairs on real matches, then true matches the matcher scored too low.

In [ ]:
n_true = report["per_entity"].set_index("s1_id")["n_true"]
view = explore.pair_view(oof.assign(selected=oof_mask), train)
cols = ["s1_id", "t_id", "p", "s1_name", "t_name", "s1_address", "t_address", "s1_country", "t_country"]
fp = view[view["selected"] & (view["y"] == 0)]
fp_single = fp[fp["s1_id"].map(n_true) == 0].sort_values("p", ascending=False)
print(f"false-positive pairs: {len(fp)}  (on true singletons: {len(fp_single)})")
display(fp_single[cols].head(12))
display(fp[fp["s1_id"].map(n_true) > 0].sort_values("p", ascending=False)[cols].head(8))
fn = view[~view["selected"] & (view["y"] == 1)].sort_values("p")
print(f"true pairs in the candidate set but not selected: {len(fn)}")
display(fn[cols].head(12))


## 8. Unseen-country stress test (proxy for France)
France appears only in the test set, so its performance cannot be measured directly. The closest offline proxy: train and tune on all *other* training countries, then score the held-out one. A small gap between the held-out score and the in-domain score means the pipeline does not lean on country-specific quirks.

In [ ]:
prep_train = PreparedSplit(train, idx_train, cand_train, X_train, y_train)
train_country_labels = sorted({c for c in countries_train.values() if normalize_country(c)})
if RUN_COUNTRY_HOLDOUT and len(train_country_labels) >= 2:
    rows = [country_holdout_eval(prep_train, train, cfg, c, n_folds=3) for c in train_country_labels]
    holdout = pd.DataFrame(rows).set_index("held-out country")
    by_country = report["tables"]["by_country"]["macro_F"]
    holdout.insert(3, "in-domain F0.5 (all-country CV)", holdout.index.map(by_country))
    display(holdout.round(4))
else:
    print("skipped (needs >= 2 countries in train, and RUN_COUNTRY_HOLDOUT=True)")


## 9. Final model → test predictions → submission files
The same index → blocking → features chain runs on the test split (TF-IDF statistics are refit on the test records, unsupervised). The saved bundle holds the fold models, the calibrator, the chosen policy and the config.

In [ ]:
trained = TrainedPipeline(cfg, matcher, policy, validation={
    "crossfit_macro_F0.5": crossfit_f, "policy": policy.describe(),
    "headline": report["headline"].to_dict(), "blocking": brep["headline"].to_dict(), "model": matcher.cv_info})
trained.save(ARTIFACT_DIR / "model_bundle.pkl")
policy_summary.to_csv(ARTIFACT_DIR / "policy_summary.csv", index=False)
matcher.importance.to_csv(ARTIFACT_DIR / "feature_importance.csv")

res = predict(test, trained)
m_path, c_path, errors, warn = write_outputs(test, res, OUTPUT_DIR, DATASET_DIR)
assert not errors, errors


Sanity checks on the test predictions. There is no test truth, but the share of entities with a match and the probability profile per country should look like the training countries; a test-only country that behaves very differently deserves a closer look (samples below).

In [ ]:
test_country = dict(zip(test.s1["entity_id"], test.s1["country"]))
per_entity = pd.DataFrame({
    "country": test.s1["country"].values,
    "n_candidates": [len(res.candidates.get(s, ())) for s in test.s1["entity_id"]],
    "n_matches": [len(res.matches.get(s, ())) for s in test.s1["entity_id"]]})
test_view = per_entity.groupby("country").agg(entities=("n_matches", "size"),
                                              share_with_match=("n_matches", lambda x: (x > 0).mean()),
                                              mean_matches=("n_matches", "mean"),
                                              mean_candidates=("n_candidates", "mean"))
train_share = pd.Series({c: np.mean([len(train.truth[s]) > 0 for s, cc in countries_train.items() if cc == c])
                         for c in set(countries_train.values())}, name="train: share with a true match")
display(test_view.join(train_share).round(3))
plots.max_prob_by_group(res.scored, test_country)
plt.show()
if unseen_countries:
    s = explore.pair_view(res.scored[res.scored["s1_id"].map(lambda x: normalize_country(test_country[x])).isin(unseen_countries)], test)
    show = ["s1_id", "t_id", "p", "s1_name", "t_name", "s1_address", "t_address"]
    print("test-only country: most confident pairs")
    display(s.sort_values("p", ascending=False)[show].head(8))
    print("test-only country: borderline pairs (probability 0.3-0.7)")
    display(s[(s["p"] > 0.3) & (s["p"] < 0.7)].sort_values("p", ascending=False)[show].head(8))


## 10. Summary and submission package
Upload `output/matching_results.tsv` to the portal. For the final zip, set `MAKE_ZIP = True` (and your team name): it packs `output/`, this `code/business_entity_resolution/` folder (without caches) and `Documentation_template.md` if it sits next to `dataset/`. Also run the official validator from `student_resource/`:

```
python3 utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test
```

In [ ]:
h = report["headline"]
print(f"blocking pair recall      : {brep['headline']['pair recall (true pairs retrieved)']:.4f} "
      f"({brep['headline']['candidates per S1 (mean)']:.1f} candidates per S1)")
print(f"cross-fitted macro F0.5   : {crossfit_f:.4f}   (policy: {policy.describe()})")
print(f"singleton accuracy (OOF)  : {h['singleton_accuracy (empty predicted)']:.4f}")
print(f"files                     : {m_path}\n                            {c_path}")
print(f"total runtime             : {(time.time() - T_START) / 60:.1f} min")

MAKE_ZIP, TEAM_NAME = False, "team_name"
if MAKE_ZIP:
    code_dir = SRC_DIR.parent
    zpath = ROOT / f"{TEAM_NAME}_submission.zip"
    skip = {"__pycache__", ".ipynb_checkpoints"}
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in (m_path, c_path):
            z.write(f, f"output/{Path(f).name}")
        for f in code_dir.rglob("*"):
            if f.is_file() and not (skip & set(f.parts)):
                z.write(f, f"code/business_entity_resolution/{f.relative_to(code_dir).as_posix()}")
        doc = None
        for cand in [ROOT / "Documentation_template.md", DATASET_DIR.parent / "Documentation_template.md", SRC_DIR.parent.parent / "Documentation_template.md"]:
            if cand.exists():
                doc = cand
                break
        if doc and doc.exists():
            z.write(doc, "Documentation_template.md")
        else:
            print("note: Documentation_template.md not found - add it to the zip yourself if required")
    print("wrote", zpath)
